In [1]:
import numpy as np
import pandas as pd


In [2]:
df= pd.read_csv("powerplant _dataset.csv")

In [ ]:
df.head()
# AT- TEMPERATURE
# V- VACCUM
# AP-PRESSURE
# RH- HUMIDITY
# PE- PRODUCE ENERGY

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [ ]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [4]:
X= df.drop("PE",axis=1)
y=df['PE']

In [5]:
X

,AT,V,AP,RH
0,8.34,40.77,1010.84,90.01
1,23.64,58.49,1011.40,74.20
2,29.74,56.90,1007.15,41.91
3,19.07,49.69,1007.22,76.79
4,11.80,40.66,1017.13,97.20
...,...,...,...,...
9563,15.12,48.92,1011.80,72.93
9564,33.41,77.95,1010.30,59.72
9565,15.99,43.34,1014.20,78.66
9566,17.65,59.87,1018.58,94.65


In [7]:
# Split out the data
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2, random_state=42)

In [8]:
df.shape

(9568, 5)

In [9]:
from sklearn.preprocessing import StandardScaler

scalar= StandardScaler()

x_train_scaled= scalar.fit_transform(X_train)
x_test_scaled= scalar.transform(X_test)


### Convert data into tensors

In [ ]:
import torch
import torch.nn as nn
#  convert data into tensors
x_train_tesnor= torch.tensor(x_train_scaled, dtype=torch.float32)
# y are labels that are not scaled we have to use .values  .views (rows,columns)
y_train_tensor= torch.tensor(y_train.values, dtype=torch.float32).view(-1,1) 


x_test_tensor= torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor= torch.tensor(y_test.values, dtype=torch.float32).view(-1,1) 



In [ ]:
type(x_test_scaled)
type(y_train)


In [ ]:
from torch.utils.data import TensorDataset,DataLoader

train_dataset= TensorDataset(x_train_tesnor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)

#  create the dataloader
train_loader= DataLoader(train_dataset,batch_size=32,shuffle=True)

test_loader= DataLoader(test_dataset,batch_size=32,)


#### Deep Learning

In [ ]:
# define orur the ANN MODEL
class ANN(nn.Model):
    def __init(self):
        super(ANN,self).__init__()

        self.model= nn.Sequential(
            # 1st Hidden Layer
            nn.Linear(X_train.shape[1],6 ),
            nn.ReLU(),
            # 2nd Hidden Layer
            nn.Linear(6,6 ),
            nn.ReLU(),
    
            # Final Layer
            nn.Linear(6,1)
            )
    def forward(self,x):
        return self.model(x)
    # backward Porpagation pytorch already handle for us- Autograd


In [ ]:
import  torch.optim as optim

model=ANN()
# loss optimizer
criterion=nn.MSELoss()
optimizer=optim.Adam(model.parameters())



In [ ]:
# Train the ANN

train_losses=[]
valid_loss=[]

best_val_loss=float("inf")
epochs=100
for epoch in range(epochs):
    model.train()
    running_loss=0.0 # training loss total for one epoch
    #  for one batch
    for xb,yb in train_loader: 
        # xb- features of one batch  yb - labels of one batch

        optimizer.zero_grad() # to remove the old gradients
        
        outputs=model(xb) # forward propagation 
    
        loss=criterion(outputs,yb)# predicted outputs,actual labels -> comput loss
    
        loss.backward()# backpropagation --- computer gradients
        optimizer.step() # parms update
    
        running_loss+=loss.item()# tensor -> float values
    epoch_train_loss= running_loss/len(train_loader)
    train_losses.append(epoch_train_loss)
     
    # Validation part
    model.eval()
    running_val_loss =0.0

    with torch.no_grad(): # no compute gradients 

        for xb,yb in test_loader:
            outputs=model(xb)
            loss= criterion(outputs,yb)
            running_val_loss+=loss 
            # we do not need of calculating the gradients
    epoch_val_loss= running_loss/len(test_loader)
    valid_loss.append(epoch_val_loss)



    #  for each epochwe are trainign the NN - FP, BP
    # and also validating that on test data
    print(f"epoch {epoch+1}/{epochs} --> trainign loss {epoch_train_loss} & validation loss {epoch_val_loss}")

#  best model loss 
    if (epoch_val_loss<best_val_loss):
        best_val_loss=epoch_val_loss
        torch.save(model.state_dict(),"bestmodel.pt") # current model state is stored 


In [ ]:
import matplotlib.pyplot as plt

loss_df=pd.DataFrame({
    "training Loss":train_losses,
    "validation Loss":val_losses
})
plt.figure(figsize=(12,8))
plt.plot(loss_df['training Loss'],label='Training Loss')
plt.plot(loss_df['validation Loss'],label='Validation Loss')

plt.xlabel("Epoches")
plt.ylabel("Losees")
plt.legend()

In [ ]:
# Loading the best model
model.load_state_dict(torch.load("bestmodel.pt")) # gives a state where val loss minimze 

In [ ]:
# Evaluate the model
model.eval()
with torch.no_grad():
    train_preds=model(x_train_tesnor)
    test_preds=model(x_test_tensor)

    train_mse_loss= criterion(train_preds,y_test_tensor)
    test_mse_loss= criterion(test_preds,y_test_tensor)

print(f"Training MSE ->  {train_mse_loss.item()}")
print(f"Testing mse loss -> {test_mse_loss.item()}")


In [ ]:
from sklearn.metrics import r2_score
print(f"R-2 SCORE -> {r2_score(y_test,test_preds)}")

In [ ]:
pred_df=pd.DataFrame(test_preds.numpy(), columns=["predicted vales"] )
actual_df=pd.DataFrame(y_test.numpy(), columns=["actual vales"] )

pd.concat([pred_df,actual_df],axis=1)
